In [1]:
import numpy as np
import pandas as pd



In [2]:
# Use Kaggle's standard input path; keep a fallback for the provided local mirror path.
from pathlib import Path

INPUT1 = Path("../input/osic-pulmonary-fibrosis-progression")
INPUT2 = Path("/kaggle/data/osic-pulmonary-fibrosis-progression")
DATA_DIR = INPUT1 if INPUT1.exists() else INPUT2

train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
sample_sub = pd.read_csv(DATA_DIR / "sample_submission.csv")



In [3]:
from tqdm import tqdm

# Expand training pairs exactly as in the original logic, but avoid chained assignment pitfalls.
train_exp = pd.DataFrame()

for patient in tqdm(train.Patient.unique()):
    df = train.loc[train.Patient == patient, :].sort_values("Weeks")

    for idx, week, percent in zip(df.index, df.Weeks, df.Percent):
        # Positive direction (future relative to baseline row idx)
        temp_df_pos = df.loc[idx:, :"SmokingStatus"].copy()
        temp_df_pos["Percent"] = percent
        temp_df_pos["Weeks"] = week
        temp_df_pos["target"] = temp_df_pos["FVC"]
        temp_df_pos["delta"] = df.loc[idx:, "Weeks"] - df.loc[idx, "Weeks"]
        temp_df_pos["FVC"] = df.loc[idx, "FVC"]

        # Negative direction (past relative to baseline row idx)
        temp_df_neg = df.loc[:idx, :"SmokingStatus"].copy()
        temp_df_neg["Weeks"] = week
        temp_df_neg["Percent"] = percent
        temp_df_neg["target"] = temp_df_neg["FVC"]
        temp_df_neg["delta"] = df.loc[:idx, "Weeks"] - df.loc[idx, "Weeks"]
        temp_df_neg["FVC"] = df.loc[idx, "FVC"]

        train_exp = pd.concat([train_exp, temp_df_pos, temp_df_neg], axis=0)

# Keep the original filtering semantics
train_exp = (
    train_exp[train_exp.delta != 0]
    .drop_duplicates()
    .dropna(axis=0)
    .reset_index(drop=True)
)



100%|██████████| 158/158 [00:01<00:00, 85.32it/s]


In [4]:
from sklearn.model_selection import train_test_split
from sklearn.compose import make_column_transformer
from sklearn.preprocessing import MinMaxScaler, OrdinalEncoder
from sklearn.metrics import mean_squared_error
from sklearn.ensemble import GradientBoostingRegressor


def laplace_log_likelihood(actual_fvc, predicted_fvc, confidence, return_values=False):
    sd_clipped = np.maximum(confidence, 70)
    delta = np.minimum(np.abs(actual_fvc - predicted_fvc), 1000)
    metric = -np.sqrt(2) * delta / sd_clipped - np.log(np.sqrt(2) * sd_clipped)
    return metric if return_values else np.mean(metric)


X = train_exp.drop(["Patient", "target"], axis=1)
y = train_exp["target"]

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, shuffle=True
)

transformer = make_column_transformer(
    (MinMaxScaler(), ["FVC", "Percent", "Age", "Weeks", "delta"]),
    (OrdinalEncoder(), ["Sex", "SmokingStatus"]),
    remainder="passthrough",
)

X_train_t = transformer.fit_transform(X_train)
X_val_t = transformer.transform(X_val)



In [5]:
# Train quantile models for prediction interval + squared error model for point prediction.
alpha = 0.95

clf = GradientBoostingRegressor(
    loss="quantile",
    alpha=alpha,
    n_estimators=250,
    max_depth=3,
    learning_rate=0.1,
    min_samples_leaf=9,
    min_samples_split=9,
    random_state=42,
)

clf.fit(X_train_t, y_train)
y_upper_train = clf.predict(X_train_t)
y_upper_val = clf.predict(X_val_t)

clf.set_params(alpha=1.0 - alpha)
clf.fit(X_train_t, y_train)
y_lower_train = clf.predict(X_train_t)
y_lower_val = clf.predict(X_val_t)

# BUGFIX: sklearn>=1.0 renamed 'ls' -> 'squared_error'
clf.set_params(loss="squared_error")
clf.fit(X_train_t, y_train)
y_pred_train = clf.predict(X_train_t)
y_pred_val = clf.predict(X_val_t)

confidence_train = (y_upper_train - y_lower_train) / 2.0
confidence_val = (y_upper_val - y_lower_val) / 2.0

# Make confidence valid (non-negative); clipping to >=70 is metric-consistent.
confidence_train = np.maximum(confidence_train, 70)
confidence_val = np.maximum(confidence_val, 70)

print("Train RMSE score: ", np.sqrt(mean_squared_error(y_train, y_pred_train)))
print("Val   RMSE score: ", np.sqrt(mean_squared_error(y_val, y_pred_val)))
print(
    "Train OSIC score: ",
    laplace_log_likelihood(y_train, y_pred_train, confidence_train),
)
print("Val   OSIC score: ", laplace_log_likelihood(y_val, y_pred_val, confidence_val))



Train RMSE score:  186.87462563838105
Val   RMSE score:  190.18541908346106
Train OSIC score:  -6.578713234485794
Val   OSIC score:  -6.619188555998227


In [6]:
# Build expanded test grid exactly as original (stamps from -12 to 133).
new_test = pd.DataFrame()
for i in np.arange(-12, 134, 1):
    temp_df = test.copy()
    temp_df["stamps"] = i
    temp_df["delta"] = temp_df["Weeks"] + temp_df["stamps"]
    new_test = pd.concat([new_test, temp_df], axis=0)

new_test.reset_index(drop=True, inplace=True)
new_test["Patient_Week"] = new_test["Patient"] + "_" + new_test["stamps"].astype(str)

X_test = new_test.drop(["Patient", "stamps", "Patient_Week"], axis=1)
X_test_t = transformer.transform(X_test)



In [7]:
# Refit the same models on full transformed training data for final predictions.
# (Core logic preserved: quantile interval + squared error point model.)
alpha = 0.95

clf = GradientBoostingRegressor(
    loss="quantile",
    alpha=alpha,
    n_estimators=250,
    max_depth=3,
    learning_rate=0.1,
    min_samples_leaf=9,
    min_samples_split=9,
    random_state=42,
)

clf.fit(X_train_t, y_train)
y_upper_test = clf.predict(X_test_t)

clf.set_params(alpha=1.0 - alpha)
clf.fit(X_train_t, y_train)
y_lower_test = clf.predict(X_test_t)

clf.set_params(loss="squared_error")
clf.fit(X_train_t, y_train)
y_pred_test = clf.predict(X_test_t)

confidence_test = (y_upper_test - y_lower_test) / 2.0
confidence_test = np.maximum(confidence_test, 70)

pred_df = pd.DataFrame(
    {
        "Patient_Week": new_test["Patient_Week"].values,
        "FVC": y_pred_test,
        "Confidence": confidence_test,
    }
)



In [8]:
# Ensure submission matches sample_submission Patient_Week set & order.
# This guarantees a valid Kaggle submission even if our expanded grid differs.
sub = sample_sub[["Patient_Week"]].merge(pred_df, on="Patient_Week", how="left")

# Fallback: if any Patient_Week is missing (shouldn't happen), fill with baseline FVC and min confidence.
missing = sub["FVC"].isna()
if missing.any():
    base_map = test.assign(
        Patient_Week=test["Patient"] + "_" + test["Weeks"].astype(str)
    ).set_index("Patient_Week")["FVC"]
    sub.loc[missing, "FVC"] = (
        sub.loc[missing, "Patient_Week"].map(base_map).fillna(test["FVC"].median())
    )
    sub.loc[missing, "Confidence"] = 70.0

# Cast/clip for safety
sub["FVC"] = sub["FVC"].astype(float)
sub["Confidence"] = np.maximum(sub["Confidence"].astype(float), 70.0)

sub.to_csv("submission.csv", index=False)
sub.head()



,Patient_Week,FVC,Confidence
0,ID00126637202218610655908_-3,2267.178011,320.373305
1,ID00126637202218610655908_-2,2230.226967,320.373305
2,ID00126637202218610655908_-1,2234.330531,320.373305
3,ID00126637202218610655908_0,2234.330531,319.097328
4,ID00126637202218610655908_1,2234.330531,319.097328


In [9]:
# Display final submission shape/columns for sanity.
print(sub.shape)
print(sub.columns.tolist())
sub

(1908, 3)
['Patient_Week', 'FVC', 'Confidence']


,Patient_Week,FVC,Confidence
0,ID00126637202218610655908_-3,2267.178011,320.373305
1,ID00126637202218610655908_-2,2230.226967,320.373305
2,ID00126637202218610655908_-1,2234.330531,320.373305
3,ID00126637202218610655908_0,2234.330531,319.097328
4,ID00126637202218610655908_1,2234.330531,319.097328
...,...,...,...
1903,ID00047637202184938901501_98,3061.842975,459.001059
1904,ID00047637202184938901501_99,3061.842975,459.001059
1905,ID00047637202184938901501_100,3061.842975,459.001059
1906,ID00047637202184938901501_101,3061.842975,459.001059
